# Alien Manual spike: run this FIRST

Run this notebook once on a **free Colab T4** (Runtime > Change runtime type > T4 GPU) before
building anything else. It answers three questions and writes `results/spike.json`:

| Question | What the notebook does | Fallback if it fails |
|---|---|---|
| **AS-1** Does the pinned stack (Python 3.12, torch 2.11, `trl==1.7.1`, AgentTune) install and run one GRPO step? | installs it, imports it, runs one GRPO step on Tiny Aya | train with plain TRL + peft and drop AgentTune |
| **AS-2** Does Tiny Aya train in fp16 with LoRA without NaN loss? | 20 SFT steps, logs every loss | 4-bit QLoRA or a smaller batch |
| **AS-3 / B-1** What do the reward function and the trainer really pass in? | prints `completions[0]` and `kwargs.keys()` on the first GRPO step | score the final tool call from the completion text alone |

How to run: make your two edits (the repo URL in the checkout cell, and a `HF_TOKEN` secret for the login cell), then
Runtime > Run all. If the install cell asks you to restart the session, restart and run all
again from the checkout cell (every cell can be re-run). Last cell downloads `results/spike.json`; put it
in the repo's `results/` folder and commit it.

Anything the author could not verify without a GPU is marked **VERIFY ON COLAB** next to the
exact line.

Sources used for install names and APIs (all read-only):
- AgentTune on PyPI: https://pypi.org/project/agenttune/
- AgentTune on GitHub (published by Lexsi-Labs, not Cohere Labs as the brief assumed): https://github.com/Lexsi-Labs/AgentTune
- Tiny Aya Global model card (gated: accept the licence on that page with your HF account first): https://huggingface.co/CohereLabs/tiny-aya-global
- TRL GRPO trainer docs (reward function signature): https://huggingface.co/docs/trl/main/en/grpo_trainer

In [ ]:
# check the runtime. Expect a Tesla T4 and Python 3.12.
import platform
print("python", platform.python_version())
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# get this repo's code (simplest option: git clone).
# EDIT ME: put your repo URL and the branch that has alien/, scripts/spike.py and
# scripts/check_results.py. The repo must be public, or upload a zip of it to /content/repo.
import os
import subprocess
import sys

REPO_URL = "https://github.com/ritamgh/alien-manual.git"
BRANCH = "main"

if not os.path.isdir("/content/repo"):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/repo"],
        check=True,
    )
os.chdir("/content/repo")
sys.path[:0] = [os.getcwd(), os.path.join(os.getcwd(), "scripts")]
print("working in", os.getcwd())

In [ ]:
# install (AS-1). Verified on Colab (Python 3.13, torch 2.14.1): plain trl==1.7.1 + peft + datasets
# install cleanly. Do NOT pip install agenttune with its dependencies: it pins torch==2.11.0 and
# about 60 other packages, and that install failed or hung on a free T4. AgentTune is tried below
# with --no-deps as an optional extra; the next cell records whether it imports.
!pip install -q "trl==1.7.1" peft datasets
!pip install -q --no-deps agenttune || true


**If AS-1 fails** (the install cell errors, or the next cell shows `install_ok = False`):
drop AgentTune. Install plain `trl==1.7.1` + `peft` only (`!pip install -q "trl==1.7.1" peft datasets`),
re-run from the next cell, and tell the team: the training plan uses plain TRL and peft
(PRD AS-1 impact).

In [ ]:
# Hugging Face login. Reads the secret HF_TOKEN from Colab Secrets (key icon in the left
# sidebar: add a secret named HF_TOKEN, switch "Notebook access" on). Falls back to a hidden
# prompt. The value is only put in os.environ: it is never printed, echoed or written to a file.
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass

    os.environ["HF_TOKEN"] = getpass("Hugging Face access value (hidden): ")
login_ready = bool(os.environ.get("HF_TOKEN"))
print("HF login ready:", login_ready)

In [ ]:
# import the stack and record versions (AS-1).
import importlib
import platform

import spike

PACKAGES = ["torch", "transformers", "trl", "peft", "accelerate", "datasets", "agenttune"]
OPTIONAL = {"agenttune"}  # DEC-4: AgentTune is an optional extra, not required
INSTALL_OK = True
for name in PACKAGES:
    try:
        importlib.import_module(name)
    except Exception as exc:  # a failed import of a required package means the stack is not usable
        if name in OPTIONAL:
            print("optional package not importable:", name, type(exc).__name__)
        else:
            INSTALL_OK = False
            print("IMPORT FAILED:", name, type(exc).__name__, exc)

PACKAGE_VERSIONS = spike.collect_package_versions(PACKAGES)
print(PACKAGE_VERSIONS)
# The PRD pins Python 3.12, torch 2.11 and trl 1.7.1; flag any mismatch.
if not platform.python_version().startswith("3.12"):
    print("WARNING: PRD pins Python 3.12, this runtime has", platform.python_version())
if not PACKAGE_VERSIONS["torch"].startswith("2.11"):
    print("WARNING: PRD pins torch 2.11, got", PACKAGE_VERSIONS["torch"])
if PACKAGE_VERSIONS["trl"] != "1.7.1":
    INSTALL_OK = False
print("INSTALL_OK =", INSTALL_OK)

In [ ]:
# AS-2. 20 fp16 LoRA SFT steps on Tiny Aya with TRL SFTTrainer. Watch the loss column.
import gc

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL_ID = "CohereLabs/tiny-aya-global"
SEED = 0

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# VERIFY ON COLAB: transformers 5 renamed torch_dtype to dtype; use torch_dtype if this errors.
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map={"": 0})

sft_dataset = Dataset.from_list(spike.build_sft_texts(n=64, seed=SEED))
sft_config = SFTConfig(
    output_dir="/content/spike_sft",
    max_steps=20,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    max_length=512,  # VERIFY ON COLAB: older TRL called this max_seq_length
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
lora = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM")
sft_trainer = SFTTrainer(
    model=model, args=sft_config, train_dataset=sft_dataset, processing_class=tokenizer,
    peft_config=lora,
)
# fp16 mixed precision needs fp32 trainable weights; peft usually does this already.
for param in sft_trainer.model.parameters():
    if param.requires_grad and param.dtype == torch.float16:
        param.data = param.data.float()

sft_trainer.train()
SFT_LOSSES = [
    float(entry["loss"]) for entry in sft_trainer.state.log_history if "loss" in entry
]
print("SFT losses:", SFT_LOSSES)
print("NaN or inf loss:", spike.loss_is_nan(SFT_LOSSES))

del sft_trainer, model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# AS-1 + AS-3 / B-1. One GRPO step, 4 generations per prompt. The reward function
# prints completions[0] and kwargs.keys() on its first call and stores them for the results file.
from trl import GRPOConfig, GRPOTrainer

TASKS = spike.tasks_by_id(n=8, seed=SEED)
grpo_dataset = Dataset.from_list(spike.build_grpo_rows(n=8, seed=SEED))

try:
    from alien.reward import exact_match_reward as score_completion
except ImportError:  # alien/reward.py is not in this checkout: use the tiny stand-in
    score_completion = spike.fallback_exact_match

FIRST_STEP = {}


def reward_func(completions, **kwargs):
    # TRL passes prompts, completions and every dataset column (here task_id) as keywords.
    if not FIRST_STEP:
        print("completions[0]:", completions[0])
        print("kwargs.keys():", kwargs.keys())
        FIRST_STEP.update(spike.capture_first_step(completions, kwargs))
    return [
        float(score_completion(TASKS[task_id], completion))
        for task_id, completion in zip(kwargs["task_id"], completions)
    ]


GRPO_STEP_OK = False
try:
    # VERIFY ON COLAB: the AgentTune route (agenttune.core.backend_factory.create_agentic_trainer)
    # is not run here: its docs show no example without tools. We run the TRL trainer it wraps.
    grpo_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map={"": 0}
    )
    grpo_config = GRPOConfig(
        output_dir="/content/spike_grpo",
        max_steps=1,
        num_generations=4,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        max_completion_length=128,
        learning_rate=1e-5,
        fp16=True,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
        seed=SEED,
    )
    grpo_trainer = GRPOTrainer(
        model=grpo_model, reward_funcs=reward_func, args=grpo_config,
        train_dataset=grpo_dataset, processing_class=tokenizer, peft_config=lora,
    )
    grpo_trainer.train()
    GRPO_STEP_OK = bool(FIRST_STEP)
except Exception as exc:  # the spike records the failure instead of stopping here
    print("GRPO STEP FAILED:", type(exc).__name__, exc)
print("GRPO_STEP_OK =", GRPO_STEP_OK)
print("captured:", FIRST_STEP)

In [ ]:
# write results/spike.json and validate it. write_results refuses an invalid file, which
# includes a NaN loss or a failed GRPO step; the printed messages then say which fallback to use.
import json
import subprocess

sha = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip() or None
first_step = FIRST_STEP or spike.capture_first_step(
    ["(the reward function was never called)"], {"(never called)": None}
)
doc = spike.build_spike_results(
    install_ok=INSTALL_OK,
    sft_losses=SFT_LOSSES,
    grpo_step_ok=GRPO_STEP_OK,
    first_step=first_step,
    python_version=platform.python_version(),
    package_versions=PACKAGE_VERSIONS,
    seed=SEED,
    git_sha=sha,
)
print(json.dumps({k: v for k, v in doc.items() if k != "completions_0"}, indent=2))
try:
    spike.write_results("results/spike.json", doc)
except ValueError as exc:
    print(exc)

In [ ]:
# validate with the repo's checker, then download the file.
!python scripts/check_results.py results/spike.json
try:
    from google.colab import files

    files.download("results/spike.json")
except Exception as exc:
    print("download skipped:", type(exc).__name__)

## What the answers mean

- `install_ok` false: **AS-1 fallback**, train with plain TRL + peft and drop AgentTune.
- `fp16_nan_loss` true (the checker prints "NaN loss in fp16"): **AS-2 fallback**, load the
  model in 4-bit (QLoRA, `bitsandbytes`) or use a smaller batch / sequence length, then re-run.
- `grpo_step_ok` false: **R-3**, plan for rejection-sampling fine-tuning instead of GRPO.
- `reward_sees_full_conversation` false: **AS-3 fallback**, score the final tool call from the
  completion text alone. True means the prompt messages reach the reward function through
  `kwargs["prompts"]` (completions themselves are usually only the new assistant message).
- Paste `completions_0` and `kwargs_keys` into the team notes: they are the real B-1 evidence.